<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/12-case-studies-and-capstone/04-deepfake-image-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study D — Deepfake detection for EU AI Act Art. 50(4): detector, VLM, or both?

**Goal:** Compare four ways to flag **undisclosed deepfakes**, the thing Article 50(4) of the EU AI Act actually regulates: an off-the-shelf deepfake **detector**, a vision-language model (**VLM**) end to end, a **provenance** check on the file, and a **hybrid** of all three. Then point the same comparison at any image you like.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **The thesis, up front:** "is this image AI-generated?" and "does this image break Art. 50(4)?" are different questions. The first is perception, where a trained detector should beat a general model. The second also needs *meaning* (does it depict a real person or event? is it evidently satire?) and *provenance* (was it already disclosed?), which no pixel classifier answers. So the likely winner is a hybrid. This notebook shows why on a handful of hand-picked cases, one at a time, so you can see every decision.

Multimodal stays off the core spine ([curriculum decision #3](../docs/curriculum-plan.md)); this is a case study that composes the evals ideas ([02](../02-evals-basics/01-measuring-outputs.ipynb), [04](../04-evals/01-golden-sets.ipynb)) with one new modality. It leaves the repo's "raw API, no frameworks" rule for one part on purpose: the detectors are ordinary image classifiers, so they load with `transformers`, the tool you'd actually reach for.

## Setup

Self-contained. Get a free key at [console.groq.com](https://console.groq.com/); in Colab add it via the **key icon** → secret `GROQ_API_KEY`. (Full walkthrough: [00-setup](../00-setup/00-environment.ipynb).) An `HF_TOKEN` secret (a free Hugging Face read token) is optional; it raises Hugging Face's rate limits.

The VLM is Groq's image-capable `qwen/qwen3.8-27b` (up to 3 images and 20 MB per request, as of 2026-10). If calls 404, check [console.groq.com/docs/vision](https://console.groq.com/docs/vision).

Runs on a CPU runtime. A full run downloads two detectors (~0.7 GB) and 7 example images, and makes about 10 VLM calls (~1,600 tokens each), well inside Groq's free daily limit.

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git@v0.1.0"

In [ ]:
from aien import setup
client, TEXT_MODEL = setup()
VISION_MODEL = "qwen/qwen3.8-27b"

## The ask, and what the law actually says

A trust-and-safety team: *"Article 50 applies from 2 August 2026. Find the deepfakes on our platform that should have been labelled and weren't."* Before building anything, read the article ([11/01 scoping](../11-customer-craft/01-scoping-and-discovery.ipynb)). Article 50 splits the duties:

| Paragraph | Who | Duty |
|---|---|---|
| 50(2) | **providers** of generative systems | mark outputs in a machine-readable, detectable way |
| 50(3) | deployers of emotion recognition / biometric categorisation | inform the people exposed |
| **50(4)** | **deployers** of deepfakes | disclose that the content is generated or manipulated; evidently artistic, satirical or fictional works only need a limited disclosure |

A **deepfake** (Art. 3(60)) is generated or manipulated content that resembles existing persons, objects, places, entities or events and would falsely appear authentic. So an image is a 50(4) problem only when **all three** hold:

1. it is AI-generated or AI-manipulated (*perception*);
2. it depicts a real, identifiable entity or event, and is not evidently fiction or satire (*meaning*);
3. the published file carries **no disclosure**: no visible label, no machine-readable marking (*provenance*).

The Commission's June 2026 Code of Practice on marking and labelling asks providers to layer at least two machine-readable marks (e.g. metadata plus watermark), so condition 3 is partly checkable from the file itself.

**Scope for this case:** still images only (50(4) also covers audio and video). The system **flags for human review**; deciding that something is a violation is a legal call, and nothing here is legal advice.

## The tools

Three components, each answering the condition it's suited to, then four systems built from them. First, shared helpers: fetching bytes politely (Hugging Face throttles bursts, so honour `Retry-After` and back off), a fixed downscaled view of an image, and a small on-disk cache so a rerun never repeats a paid call.

In [ ]:
import base64, hashlib, io, json, os, re, time, urllib.error, urllib.parse, urllib.request
from PIL import Image, ImageDraw, ImageFont
from PIL.PngImagePlugin import PngInfo

MAX_SIDE = 512                     # every model sees the image downscaled to this
MAX_TOKENS = 1024                  # headroom for models that reason before answering
CACHE_PATH = "case_d_cache.json"   # model outputs only
SAMPLE_DIR = "case_d_samples"      # the downloaded example images; see the license note below

def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:              # not in Colab, or no HF_TOKEN secret: anonymous is fine
        return None

def http_get(url, headers=None, tries=5):
    """GET with backoff. On 429 (throttled) wait as long as the server asks."""
    headers = {"User-Agent": "ai-engineer-notebooks (case study D)", **(headers or {})}
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code not in (429, 500, 502, 503, 504) or attempt == tries:
                raise
            ra = e.headers.get("Retry-After") or ""
            wait = int(ra) if ra.isdigit() else 10 * attempt
            print(f"   HTTP {e.code}; waiting {wait}s ({attempt}/{tries})")
        except (urllib.error.URLError, TimeoutError) as e:
            if attempt == tries:
                raise
            wait = 10 * attempt
            print(f"   {type(e).__name__}; waiting {wait}s ({attempt}/{tries})")
        time.sleep(wait)

def view(raw, side=MAX_SIDE):
    """The pixels every model sees: decoded, RGB, downscaled. File metadata doesn't survive this."""
    img = Image.open(io.BytesIO(raw))
    img.draft("RGB", (side, side))     # JPEGs decode straight at reduced size
    img = img.convert("RGB")
    img.thumbnail((side, side))
    return img

CACHE = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}

def cached(key, fn):
    if key not in CACHE:
        CACHE[key] = fn()
        with open(CACHE_PATH, "w") as f:
            json.dump(CACHE, f)
    return CACHE[key]

### D — provenance check

The cheapest check, and the only one that answers condition 3 from the file: is there a machine-readable AI mark? We look for the IPTC `trainedAlgorithmicMedia` term (the vocabulary provenance standards use) and for an embedded C2PA manifest (its JUMBF box). It reads the **original bytes**: any re-encode, like `view()` above or a screenshot, throws the mark away. Presence of a C2PA box is not *verification*: that needs the signature checked against a trust list (exercise 5). SynthID-style invisible watermarks can't be checked here at all.

In [ ]:
def provenance(raw):
    return {"ai_metadata": b"trainedAlgorithmicMedia" in raw,
            "c2pa_manifest": b"jumb" in raw and b"c2pa" in raw}

### A — off-the-shelf detectors

You don't have to train a detector to use one. Two pretrained image classifiers, both used as-is with a 0.5 threshold:

- **`A_openfake`**: the SwinV2-base detector the [OpenFake](https://huggingface.co/datasets/ComplexDataLab/OpenFake) authors (Complex Data Lab, McGill / Mila) ship in their demo, re-hosted as a standard `transformers` model at [`bendavidsteel/openfake-swinv2-base`](https://huggingface.co/bendavidsteel/openfake-swinv2-base). Trained on OpenFake's train split, so it knows this dataset's generators. Its card says to divide the logits by 2.0 before the softmax.
- **`A_generic`**: [`umm-maybe/AI-image-detector`](https://huggingface.co/umm-maybe/AI-image-detector), one of the most-downloaded "AI image detectors" on the Hub, trained in 2022 on older generators. It's here as the contrast: what you get by grabbing the popular model.

Each returns `p_fake` in [0, 1]. A detector answers condition 1 only.

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5

class Detector:
    def __init__(self, repo, fake_label, temperature=1.0):
        self.proc = AutoImageProcessor.from_pretrained(repo)
        self.model = AutoModelForImageClassification.from_pretrained(repo).to(DEVICE).eval()
        self.fake = {v.lower(): int(k) for k, v in self.model.config.id2label.items()}[fake_label]
        self.temperature = temperature

    @torch.no_grad()
    def __call__(self, raws):
        x = self.proc(images=[view(r) for r in raws], return_tensors="pt").to(DEVICE)
        p = (self.model(**x).logits.float() / self.temperature).softmax(-1)[:, self.fake]
        return p.cpu().tolist()

t0 = time.time()
DETECTORS = {
    "A_openfake": Detector("bendavidsteel/openfake-swinv2-base", "fake", temperature=2.0),
    "A_generic": Detector("umm-maybe/AI-image-detector", "artificial"),
}
DET = "A_openfake"                 # the detector the A and C systems use
print(f"loaded {list(DETECTORS)} in {time.time() - t0:.0f}s on {DEVICE}")

> **⚠️ Production reality —** a detector's model card is part of its spec. The OpenFake detector's card reports that on social media it flags many **real** text graphics (screenshots, news chyrons, infographics, meme macros) as AI-generated: 22 of 35 such images in their test. If your platform is full of screenshots, that's your false-flag rate, and no number on OpenFake will show it.

### B — a VLM, end to end

One call per image. The VLM answers every condition it can see: is it fake (`p_fake`), does it depict a real entity, is it evident fiction, does it carry a visible AI label. It cannot read file metadata, which is realistic: a VLM sees pixels. The `evidence` sentence is for a human reviewer and is **never scored**: a plausible reason is not a correct one.

In [ ]:
VLM_PROMPT = (
    "You audit images under EU AI Act Article 50(4). Look at the image and reply with ONLY a JSON object:\n"
    '{"p_fake": <0-1, probability the image is AI-generated or AI-manipulated; ordinary compression, '
    "resizing or filters are NOT manipulation>,\n"
    ' "real_entity": <true if it depicts an existing, identifiable person, place, object or event '
    "and could falsely appear authentic>,\n"
    ' "evident_fiction": <true if it is evidently artistic, satirical, fantastical or fictional>,\n'
    ' "visible_ai_label": <true if the image itself shows a visible notice that it is AI-generated>,\n'
    ' "evidence": "<one short sentence>"}'
)

def parse_json(text):
    m = re.search(r"\{.*\}", text or "", re.DOTALL)
    try:
        return json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        return None

def vlm_judge(raw):
    buf = io.BytesIO()
    view(raw).save(buf, format="PNG")
    t0 = time.time()
    resp = client.chat.completions.create(
        model=VISION_MODEL, max_tokens=MAX_TOKENS, temperature=0,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": VLM_PROMPT},
            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()}}]}])
    out = parse_json(resp.choices[0].message.content) or {}
    try:
        p = min(max(float(out["p_fake"]), 0.0), 1.0)
    except (KeyError, TypeError, ValueError):
        p = None                                  # unparseable: shown as such, not guessed
    return {"p_fake": p,
            **{k: out.get(k) is True for k in ("real_entity", "evident_fiction", "visible_ai_label")},
            "evidence": str(out.get("evidence", ""))[:200],
            "tokens": resp.usage.total_tokens if resp.usage else None,
            "latency": time.time() - t0}

> **⚠️ Production reality —** a generative model's `p_fake` is a *token it wrote*, not a calibrated probability: expect clustering at round numbers and confident, specific-sounding evidence on genuinely real photos. Measure the one you'd ship rather than trusting a leaderboard.

### Four systems built from the three components

| System | Flags when |
|---|---|
| **A only** | the detector says fake (the naive build: every fake is a "violation") |
| **A + D** | detector says fake **and** no metadata mark |
| **B end to end** | the VLM says fake, real entity, not fiction, no visible label |
| **C hybrid** | detector says fake **and** VLM says real entity, not fiction, no visible label **and** no metadata mark |

`judge()` runs all three components on one published file and returns every system's decision. B and C share the same VLM call, so they differ only in *who decides "is it fake"*.

In [ ]:
SYSTEMS = ["A only", "A + D", "B end to end", "C hybrid"]

def flags(det, vlm, prov):
    is_fake = det[DET] >= THRESHOLD
    meaning = vlm["real_entity"] and not vlm["evident_fiction"] and not vlm["visible_ai_label"]
    return {"A only": is_fake,
            "A + D": is_fake and not prov["ai_metadata"],
            "B end to end": (vlm["p_fake"] or 0) >= 0.5 and meaning,
            "C hybrid": is_fake and meaning and not prov["ai_metadata"]}

def judge(raw):
    key = f"vlm|{VISION_MODEL}|{hashlib.sha256(raw).hexdigest()[:16]}"   # same bytes, same call
    det = {name: d([raw])[0] for name, d in DETECTORS.items()}
    vlm = cached(key, lambda: vlm_judge(raw))
    prov = provenance(raw)
    return {"det": det, "vlm": vlm, "prov": prov, "flags": flags(det, vlm, prov)}

## The examples

**[OpenFake](https://huggingface.co/datasets/ComplexDataLab/OpenFake)** (`core` subset) has 2.4M real and generated images from 75 sources, each with its generator name and a text prompt. Two things about it shaped this section:

- **Real-entity deepfakes are rare in it.** In 206 random test-split fakes we found about five whose prompt names a real person, team or show. A random sample of 15 can easily contain none, and then there's nothing for the compliance systems to find.
- **Most "prompts" are captions.** In one page of 100 test rows, 86 have `prompt_kind = qwen_caption`: a captioning model's description of the finished image, not what the user typed. Captioners describe "a man in a suit at a desk"; they rarely name him. So a celebrity deepfake can hide behind an anonymous caption.

So instead of sampling, we **pick** the cases: two realistic fakes of real entities found in the test split, an AI image of an anonymous person, an anime illustration, and three real photos. The labels for conditions 1 and 2 are ours, by hand, which is what a small golden set should be ([04/01](../04-evals/01-golden-sets.ipynb)). Each image is fetched by its row number through Hugging Face's dataset viewer API: two small requests per image, no 3.4 TB download.

> **⚠️ Production reality —** OpenFake is **CC BY-NC 4.0** (non-commercial) and contains images of real public figures. The images are kept under `SAMPLE_DIR` so a rerun doesn't fetch them again; keep that folder private, don't redistribute it, and delete it when you're done.

In [ ]:
DATASET_ID, SUBSET, SPLIT = "ComplexDataLab/OpenFake", "core", "test"
VIEWER = "https://datasets-server.huggingface.co"   # Hugging Face's dataset viewer API
HF_HEADERS = {"Authorization": f"Bearer {hf_token()}"} if hf_token() else {}

EXAMPLES = [   # row in OpenFake core/test, what it shows, and our hand labels for condition 2
    {"row": 6831, "label": "fake", "what": "Real Madrid press conference, named player",
     "real_entity": True, "evident_fiction": False},
    {"row": 29127, "label": "fake", "what": "'The Daily Show with Trevor Noah' studio",
     "real_entity": True, "evident_fiction": False},
    {"row": 3251, "label": "fake", "what": "anonymous man, 'Bernie for President' shirt",
     "real_entity": False, "evident_fiction": False},
    {"row": 6814, "label": "fake", "what": "anime illustration on a beach",
     "real_entity": False, "evident_fiction": True},
    {"row": 6811, "label": "real", "what": "bearded dragon (ImageNet photo)",
     "real_entity": False, "evident_fiction": False},
    {"row": 6851, "label": "real", "what": "Mini Cooper under a tarp (DOCCI photo)",
     "real_entity": False, "evident_fiction": False},
    {"row": 6804, "label": "real", "what": "parking-garage mirror (DOCCI photo)",
     "real_entity": False, "evident_fiction": False},
]

def fetch_example(ex):
    """One example's image bytes, from disk if we already have them."""
    path = os.path.join(SAMPLE_DIR, f"{SPLIT}-{ex['row']}.img")
    if os.path.exists(path):
        with open(path, "rb") as f:
            return f.read()
    q = urllib.parse.urlencode({"dataset": DATASET_ID, "config": SUBSET, "split": SPLIT,
                                "offset": ex["row"], "length": 1})
    row = json.loads(http_get(f"{VIEWER}/rows?{q}", HF_HEADERS))["rows"][0]["row"]
    if str(row["label"]).lower() != ex["label"]:
        print(f"⚠️ row {ex['row']} is now labelled {row['label']!r}: the dataset changed, pick another row")
    ex["generator"] = row.get("model")
    raw = http_get(row["image"]["src"])            # a signed link to this one image
    os.makedirs(SAMPLE_DIR, exist_ok=True)
    with open(path, "wb") as f:
        f.write(raw)
    return raw

for ex in EXAMPLES:
    ex["raw"] = fetch_example(ex)
    print(f"row {ex['row']:>6}  {ex['label']:4}  {ex['what']}")

# A contact sheet, so you know what you're judging.
thumbs = [view(ex["raw"], 160) for ex in EXAMPLES]
sheet = Image.new("RGB", (165 * len(thumbs), 165), "white")
for i, t in enumerate(thumbs):
    sheet.paste(t, (165 * i, 0))
sheet

## Publish them: the disclosure is ours to set

Condition 3 is about the **published file**, so we publish each image ourselves. Most go out with no disclosure. The two real-entity fakes also go out a second time, disclosed: one with a visible "AI-generated" label drawn on the image, one with the IPTC `trainedAlgorithmicMedia` mark written into its metadata. The same picture is then a violation in one file and not in the other, which is exactly the distinction "A only" can't make.

**Gold** (what a careful human reviewer would decide) = fake **and** real entity **and** not evident fiction **and** not disclosed.

In [ ]:
XMP = ('<x:xmpmeta xmlns:x="adobe:ns:meta/"><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">'
       '<rdf:Description xmlns:Iptc4xmpExt="http://iptc.org/std/Iptc4xmpExt/2008-02-29/" '
       'Iptc4xmpExt:DigitalSourceType="http://cv.iptc.org/newscodes/digitalsourcetype/trainedAlgorithmicMedia"/>'
       '</rdf:RDF></x:xmpmeta>')

def png_bytes(img, pnginfo=None):
    buf = io.BytesIO()
    img.save(buf, format="PNG", pnginfo=pnginfo)
    return buf.getvalue()

def with_metadata(raw):
    info = PngInfo()
    info.add_itxt("XML:com.adobe.xmp", XMP)
    return png_bytes(view(raw, 1024), info)

def with_visible_label(raw, text="AI-generated"):
    img = view(raw, 1024)
    draw, size = ImageDraw.Draw(img), max(14, img.height // 18)
    try:
        font = ImageFont.load_default(size=size)
    except TypeError:                      # Pillow < 10.1
        font = ImageFont.load_default()
    x0, y0, x1, y1 = draw.textbbox((0, 0), text, font=font)
    pad = size // 3
    draw.rectangle([8, 8, 8 + x1 - x0 + 2 * pad, 8 + y1 - y0 + 2 * pad], fill="black")
    draw.text((8 + pad - x0, 8 + pad - y0), text, fill="white", font=font)
    return png_bytes(img)

PUBLISH = {"none": lambda raw: raw, "visible": with_visible_label, "metadata": with_metadata}
by_row = {ex["row"]: ex for ex in EXAMPLES}
CASES = [{**by_row[row], "disclosure": d} for row, d in [
    (6831, "none"), (6831, "visible"), (29127, "none"), (29127, "metadata"),
    (3251, "none"), (6814, "none"), (6811, "none"), (6851, "none"), (6804, "none")]]

for c in CASES:
    c["published"] = PUBLISH[c["disclosure"]](c["raw"])
    c["gold"] = (c["label"] == "fake" and c["real_entity"] and not c["evident_fiction"]
                 and c["disclosure"] == "none")
print(f"{len(CASES)} cases, {sum(c['gold'] for c in CASES)} of them violations")

## One case, step by step

Before the table, follow one case through every component: the first real-entity fake, undisclosed. If the VLM's JSON doesn't parse or a score looks inverted, you find out here for one API call.

In [ ]:
from IPython.display import display

c = CASES[0]
display(view(c["published"], 320))
r = judge(c["published"])
print(f"{c['what']}  (row {c['row']}, {c['label']}, disclosure={c['disclosure']})\n")
for name, p in r["det"].items():
    print(f"  {name:12}: p_fake {p:.3f} → {'fake' if p >= THRESHOLD else 'real'}")
v = r["vlm"]
print(f"  B VLM       : " + json.dumps({k: v[k] for k in ("p_fake", "real_entity", "evident_fiction", "visible_ai_label")}))
print(f"                evidence {v['evidence']!r}; {v['tokens']} tokens, {v['latency']:.1f}s")
print(f"  D provenance: {r['prov']}")
print(f"  gold        : {'VIOLATION' if c['gold'] else 'no violation'}")
for s in SYSTEMS:
    print(f"  {s:12}: {'FLAG' if r['flags'][s] else '-'}")

## Compare on every case

Same thing for all nine cases. Every VLM call is cached, so rerunning this cell is free. ✓ means the system agreed with gold, ✗ that it didn't.

In [ ]:
import groq

rows = []
for c in CASES:
    try:
        rows.append((c, judge(c["published"])))
    except groq.RateLimitError as e:            # the client already retried; usually the daily cap
        print(f"⏸ VLM rate limit: {str(e)[:150]}\n  Rerun this cell later; finished calls are cached.")
        break

print(f"{'case':52} {'gold':>4}  {'openfake':>8} {'generic':>8} {'vlm':>5}  " + " ".join(f"{s:>12}" for s in SYSTEMS))
for c, r in rows:
    v = r["vlm"]
    name = f"{c['what'][:40]} [{c['disclosure']}]"
    marks = " ".join(f"{('FLAG' if r['flags'][s] else '-') + (' ✓' if r['flags'][s] == c['gold'] else ' ✗'):>12}" for s in SYSTEMS)
    vp = "n/a" if v["p_fake"] is None else f"{v['p_fake']:.2f}"
    print(f"{name:52} {'yes' if c['gold'] else 'no':>4}  {r['det']['A_openfake']:>8.2f} {r['det']['A_generic']:>8.2f} {vp:>5}  {marks}")

print()
for s in SYSTEMS:
    print(f"{s:12}: {sum(r['flags'][s] == c['gold'] for c, r in rows)}/{len(rows)} cases right")

Read it column by column:

- **Detection (the three score columns).** The OpenFake detector was trained on this dataset's generators and should separate fakes from reals cleanly. The popular generic detector has never seen 2025–2026 generators; watch where it calls a fake real or a real photo fake. The VLM's `p_fake` sits in between and clusters on round numbers.
- **A only** flags every fake, so it "catches" the disclosed copies, the anonymous man and the anime image too: every one a false flag a reviewer has to clear.
- **A + D** clears the metadata-marked copy but not the visibly labelled one: it can read files, not pixels.
- **B and C** can clear the visible label and the fiction, if the VLM reads them. Where B and C disagree, it's the "is it fake" call: the detector's against the VLM's.
- The **anonymous man in a "Bernie for President" shirt** is a judgement call on purpose: a real politician is *named*, but no real person is *depicted*. We labelled it not a deepfake; if your reviewers would disagree, that's a policy decision to write down before you measure anything.

Nine cases are a demonstration, not a measurement. To choose a design you'd score hundreds of cases per condition (exercise 1).

## Try your own image

Give any image URL (or a local path, e.g. after uploading through Colab's file panel) and see what each component and system decides. There's no gold here: you're the reviewer.

Try a photo straight from your phone, an image downloaded straight from ChatGPT or Gemini (their files carry C2PA and IPTC marks, which `D` should find), and then a screenshot of that same image (the marks are gone).

> **⚠️ Privacy —** the VLM step sends the image to Groq's API. Don't use photos of private people or anything you wouldn't upload.

In [ ]:
IMAGE = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a8/Tour_Eiffel_Wikimedia_Commons.jpg/960px-Tour_Eiffel_Wikimedia_Commons.jpg"

raw = http_get(IMAGE) if IMAGE.startswith("http") else open(IMAGE, "rb").read()
display(view(raw, 320))
r = judge(raw)
for name, p in r["det"].items():
    print(f"{name:12}: p_fake {p:.3f} → {'fake' if p >= THRESHOLD else 'real'}")
v = r["vlm"]
print("VLM         :", json.dumps({k: v[k] for k in ("p_fake", "real_entity", "evident_fiction", "visible_ai_label")}))
print(f"              {v['evidence']!r}")
print("provenance  :", r["prov"])
print()
for s in SYSTEMS:
    print(f"{s:12}: {'FLAG for review' if r['flags'][s] else '-'}")

## What this should tell you

Check these against your own output rather than taking them on faith:

- **Detection is a detector's job, but only for generators it has seen.** The OpenFake detector looks excellent here because it was trained on this dataset; the popular generic one shows what happens on generators newer than its training data. Your production detector is always somewhere between the two, and drifting toward the second.
- **Compliance is not detection.** "A only" flags every fake, disclosed or not, real entity or anime. Only systems that check provenance *and* meaning can clear those.
- **Metadata is fragile.** Any re-encode strips it (try the screenshot in the last section). Visible labels and robust watermarks survive; metadata alone is not a disclosure strategy.
- **The VLM earns its place on meaning.** Real entity, satire, a visible label: these are reading tasks, where a pixel classifier has nothing to say.

> **⭐ Key takeaway —** decompose the legal question into the conditions it's made of, give each condition the cheapest tool that answers it well, and measure each piece and the whole. Here that's a detector for *is it fake*, provenance for *was it disclosed*, a VLM for *what does it depict*, and a human for *is this a violation*. A single model asked the whole question is easier to build and harder to trust.

> **🔵 Interview signal —** the hybrid also has a cost lever B lacks: run the cheap detector on everything and call the VLM **only** on images it flags. If the detector flags 5% of traffic, VLM spend drops ~20×. Say that number in the design review.

## Exercises

1. **From demo to measurement.** Build a golden set of 50+ cases per condition: real photos from two different sources, fakes from generators newer than the detector's training data, real-entity fakes (search OpenFake rows where `prompt_kind == "published_prompt"`: user-written prompts keep the names that captions drop). Report precision and recall per system ([04/01](../04-evals/01-golden-sets.ipynb)). Does the ranking from nine cases survive?
2. **Calibrate the threshold.** 0.5 is the detectors' default, not a decision. Score 100 real photos from *your* platform's kind of source and set each detector's threshold so at most 5% of them are flagged. How far does it move from 0.5?
3. **Your own photo set.** Put 20 of your own images in a folder with a CSV (`file,label,real_entity,evident_fiction`) and run them through `judge()`. Include phone photos, screenshots and AI images downloaded straight from a generator. Which system makes the most mistakes, and on what?
4. **Train your own detector.** Freeze a CLIP image encoder, embed 200 OpenFake training images, and fit a logistic regression on the embeddings (a linear probe, the UnivFD recipe). How close does it get to `A_openfake` on your golden set, and on generators neither has seen?
5. **Real C2PA.** Install the `c2pa-python` package and replace the byte check in `provenance` with actual manifest reading and signature validation. Test it on a C2PA-signed image straight from a generator. What does "manifest present but signature invalid" mean for the compliance decision?
6. **Hybrid economics.** Change `judge()` so the VLM is called only when the detector flags. At 5M uploads/month with 1% undisclosed deepfakes, what's the VLM spend for B versus the gated hybrid, and the reviewer load (flags × 1 minute)? Write the one-sentence design-doc recommendation.